# Training of the YOLO pose parking spot detection

Alternative to `train_parking_net.ipynb`: trains a YOLO pose network of [Ultralytics](https://docs.ultralytics.com/tasks/pose/) on the same CVAT annotations. Every parking spot is one object (class + bounding box) with its **four corners as keypoints**.

Expected data: the same as for `train_parking_net.ipynb` (CVAT export **"CVAT for images 1.1"** + one folder per rosbag).

Steps:
1. Load the CVAT annotations and split them by rosbag
2. Export them to the YOLO pose format (`data/yolo/<dataset>/`)
3. Train with Ultralytics (runs are saved in `runs/pose/`)
4. Test on the validation images
5. Copy the best model to `models/yolo_pose.pt` for the `parking_perception_node` (`detector: 'yolo_pose'` in `config/model.yaml`)

Requirements: `pip install -r ../requirements-yolo.txt`

## 1. Import Libraries

In [ ]:
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO

# Import the parking package without sourcing the ROS workspace
PACKAGE_DIR = Path.cwd().parent
sys.path.insert(0, str(PACKAGE_DIR))

from parking.training.dataset import bag_name, cvat_labels, load_cvat_xml  # noqa: E402
from parking.training.yolo_export import export_yolo_pose  # noqa: E402

print("Ultralytics version:", ultralytics.__version__)
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
    DEVICE = 0
else:
    print("No GPU found, using CPU instead.")
    DEVICE = "cpu"

## 2. Data Loading and Export

Same split as in `train_parking_net.ipynb`: the validation set consists of whole rosbags (with only one rosbag, the frames are split randomly). The corners are sorted into the fixed order bottom-left, bottom-right, top-right, top-left, so the keypoints have the same meaning in every parking spot.

In [ ]:
DATA_DIR = PACKAGE_DIR / "data" / "dataset_v4"
YOLO_DIR = PACKAGE_DIR / "data" / "yolo" / DATA_DIR.name
VAL_FRACTION = 0.2
SEED = 42

XML_FILES = sorted(DATA_DIR.glob("annotations*.xml"))
print("Annotations:", [f.name for f in XML_FILES])

# All tasks have to belong to the same CVAT project, i.e. have the same labels
CLASSES = cvat_labels(XML_FILES[0])
for xml_file in XML_FILES[1:]:
    if cvat_labels(xml_file) != CLASSES:
        raise ValueError(f"Different labels in {xml_file.name}")
print("Classes:", CLASSES)

samples = [s for f in XML_FILES for s in load_cvat_xml(f, DATA_DIR, CLASSES)]
print(f"{len(samples)} images, {sum(len(s.spots) for s in samples)} parking spots")
for class_id, name in enumerate(CLASSES):
    count = sum(a.class_id == class_id for s in samples for a in s.spots)
    print(f"  {name}: {count}")

# Split by rosbag (file name without the frame index)
random.seed(SEED)
bags = sorted({bag_name(s) for s in samples})
if len(bags) > 1:
    num_val_bags = max(1, round(VAL_FRACTION * len(bags)))
    val_bags = set(random.sample(bags, num_val_bags))
    train_samples = [s for s in samples if bag_name(s) not in val_bags]
    val_samples = [s for s in samples if bag_name(s) in val_bags]
    print("Validation bags:", sorted(val_bags))
else:
    shuffled = random.sample(samples, len(samples))
    num_val = max(1, round(VAL_FRACTION * len(samples)))
    val_samples, train_samples = shuffled[:num_val], shuffled[num_val:]
print(f"Train: {len(train_samples)} images, validation: {len(val_samples)} images")

DATA_YAML = export_yolo_pose({"train": train_samples, "val": val_samples}, CLASSES, YOLO_DIR)
print("Exported to", DATA_YAML)
print(DATA_YAML.read_text())

## 3. Training

Starts from a model pretrained on COCO keypoints (downloaded automatically). `n` is the smallest and fastest model, `s` is more accurate but about 3x slower on the vehicle.

Important augmentation: **`fliplr=0.0`**. On the track 90° parking spots are only on the left and parallel ones only on the right, a mirrored image would teach the network wrong combinations.

Ultralytics shows the progress and saves the curves, metrics and example images in `runs/pose/<name>/`. The training stops early if the validation metric does not improve for `patience` epochs.

In [ ]:
BASE_MODEL = "yolo11n-pose.pt"  # or yolo11s-pose.pt
IMAGE_SIZE = 640  # The BEV image (800x640) is scaled to 640 on the long side
EPOCHS = 30
BATCH_SIZE = 8
RUN_NAME = f"parking_{DATA_DIR.name}"

model = YOLO(BASE_MODEL)
model.train(
    data=str(DATA_YAML),
    imgsz=IMAGE_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    patience=30,
    device=DEVICE,
    workers=4,
    seed=SEED,
    project=str(PACKAGE_DIR / "runs" / "pose"),
    name=RUN_NAME,
    exist_ok=True,
    # Augmentation
    fliplr=0.0,  # Never mirror (side of the road matters)
    hsv_h=0.0,  # Grayscale images, no hue/saturation
    hsv_s=0.0,
    hsv_v=0.4,  # Brightness
    degrees=5.0,  # Small rotations
    mosaic=1.0,
    close_mosaic=10,  # No mosaic in the last epochs
)
RUN_DIR = Path(model.trainer.save_dir)
print("Run saved to", RUN_DIR)

In [ ]:
# Training curves of Ultralytics (losses, precision, recall, mAP of boxes and keypoints)
plt.figure(figsize=(18, 9))
plt.imshow(plt.imread(RUN_DIR / "results.png"))
plt.axis("off")
plt.show()

## 4. Testing

Validates the best model (`mAP50-95(P)` is the metric of the keypoints, `(B)` of the boxes) and draws the predictions (green, blue dot: first corner = bottom-left) and the annotations (red) into the validation images.

In [ ]:
SCORE_THRESHOLD = 0.3

best_model = YOLO(RUN_DIR / "weights" / "best.pt")
metrics = best_model.val(data=str(DATA_YAML), imgsz=IMAGE_SIZE, device=DEVICE, plots=False)
print(f"Box mAP50: {metrics.box.map50:.3f}, keypoint mAP50: {metrics.pose.map50:.3f}")

In [ ]:
num_images = min(6, len(val_samples))
rows = (num_images + 2) // 3
fig, axes = plt.subplots(rows, 3, figsize=(18, 5 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")

for idx, sample in enumerate(random.sample(val_samples, num_images)):
    result = best_model.predict(str(sample.image_path), imgsz=IMAGE_SIZE, conf=SCORE_THRESHOLD, verbose=False)[0]

    ax = axes.flat[idx]
    ax.imshow(plt.imread(sample.image_path), cmap="gray")
    for annotation in sample.spots:
        ax.fill(*annotation.corners.T, edgecolor="red", fill=False, linewidth=2)

    if result.keypoints is None:
        continue
    for class_id, score, corners in zip(
        result.boxes.cls.int().tolist(), result.boxes.conf.tolist(), result.keypoints.xy.cpu().numpy()
    ):
        ax.fill(*corners.T, edgecolor="lime", fill=False, linewidth=2)
        ax.plot(*corners[0], "o", color="blue")
        ax.text(*corners.mean(axis=0), f"{CLASSES[class_id]} {score:.2f}", color="lime")
plt.show()

## 5. Save the Model for the Node

Copies the best model to `models/yolo_pose.pt`. The installed package links to this file (`colcon build --symlink-install`), so restarting the `parking_perception_node` is enough.

In [ ]:
MODEL_PATH = PACKAGE_DIR / "models" / "yolo_pose.pt"
shutil.copy2(RUN_DIR / "weights" / "best.pt", MODEL_PATH)
print("Saved model to", MODEL_PATH)